In [1]:
import pandas as pd, numpy as np

nodes = pd.read_csv('network_nodes_with_community3.csv', sep=';', encoding='utf-8-sig')
nodes['account_type'] = nodes['account_type'].str.strip()

# tandai siapa yang punya data profil (muncul sebagai penulis tweet)
tweets = pd.read_csv('Book62.csv')
authors = set(tweets['username'].astype(str).str.lstrip('@').str.strip().str.lower())
nodes['u'] = nodes['username'].astype(str).str.lstrip('@').str.strip().str.lower()
nodes['has_profile'] = nodes['u'].isin(authors)

rng = 42  # seed untuk reproduktibilitas
parts = [
    nodes[nodes.account_type=='media/news'],
    nodes[nodes.account_type=='elite/political'],
    nodes[(nodes.account_type=='grassroot') & (nodes.has_profile)].sample(100, random_state=rng),
    nodes[(nodes.account_type=='grassroot') & (~nodes.has_profile)].sample(100, random_state=rng),
]
sample = pd.concat(parts)

# acak urutan supaya pengkode tidak menebak kategori dari urutan
sample = sample.sample(frac=1, random_state=rng).reset_index(drop=True)

# LEMBAR KODING: tanpa kolom account_type (label AI disembunyikan)
sheet = sample[['username','has_profile']].copy()
sheet['coder_label'] = ''
sheet['notes'] = ''
sheet.to_csv('coding_sheet.csv', index=False)

# KUNCI JAWABAN: disimpan terpisah, jangan dibuka sampai koding selesai
sample[['username','account_type','has_profile']].to_csv('ai_labels_HIDDEN.csv', index=False)

In [40]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, classification_report

def load(name):
    with open(name, encoding='utf-8-sig', errors='replace') as f:
        head = f.read(2048)
    sep = ';' if head.count(';') > head.count(',') else ','
    df = pd.read_csv(name, sep=sep, encoding='utf-8-sig')
    df.columns = [c.strip().lower() for c in df.columns]
    df['username'] = df['username'].astype(str).str.strip().str.lower()
    if 'coder_label' in df.columns:
        df['coder_label'] = df['coder_label'].astype(str).str.strip().str.lower()
    if 'account_type' in df.columns:
        df['account_type'] = df['account_type'].astype(str).str.strip().str.lower()
    return df

c1 = load('coding_coder1.csv')
c2 = load('coding_coder2.csv')
ai = load('ai_labels_HIDDEN.csv')

print("baris:", len(c1), len(c2), len(ai))

m = (c1[['username','coder_label']]
     .merge(c2[['username','coder_label']], on='username', suffixes=('_1','_2'))
     .merge(ai[['username','account_type','has_profile']], on='username'))
print("baris setelah merge:", len(m))

# --- cek kewarasan: apakah dua coder identik sempurna? ---
raw_agree = (m.coder_label_1 == m.coder_label_2).mean()
print(f"\nraw agreement coder1 vs coder2: {raw_agree:.3f}")
if raw_agree > 0.98:
    print(">> PERINGATAN: agreement hampir sempurna. Pastikan kedua file BENAR-BENAR")
    print(">> koding independen, bukan tersalin. Periksa sebelum lanjut.")

# --- 1) intercoder reliability ---
k = cohen_kappa_score(m.coder_label_1, m.coder_label_2)
print(f"\nCohen's kappa (coder1 vs coder2): {k:.3f}")

# --- 2) akurasi AI vs konsensus manusia ---
cons = m[m.coder_label_1 == m.coder_label_2].copy()
print(f"\nbaris konsensus (dua coder sepakat): {len(cons)} dari {len(m)}")
acc = (cons.account_type == cons.coder_label_1).mean()
print(f"akurasi label AI vs konsensus: {acc:.3f}")
print("\nlaporan per kategori (AI vs konsensus):")
print(classification_report(cons.coder_label_1, cons.account_type, zero_division=0))

# --- 3) subset tanpa data profil ---
sub = cons[cons.has_profile == False]
if len(sub):
    print(f"akurasi pada akun tanpa data profil (n={len(sub)}): {(sub.account_type==sub.coder_label_1).mean():.3f}")

baris: 437 437 437
baris setelah merge: 437

raw agreement coder1 vs coder2: 0.931

Cohen's kappa (coder1 vs coder2): 0.890

baris konsensus (dua coder sepakat): 407 dari 437
akurasi label AI vs konsensus: 0.919

laporan per kategori (AI vs konsensus):
                 precision    recall  f1-score   support

elite/political       0.98      0.91      0.95       116
      grassroot       0.95      0.89      0.92       207
     media/news       0.80      1.00      0.89        84

       accuracy                           0.92       407
      macro avg       0.91      0.93      0.92       407
   weighted avg       0.93      0.92      0.92       407

akurasi pada akun tanpa data profil (n=226): 0.876
